In [ ]:
# console


# (async function() {
#     const matchId = 16738923;
#     console.log("جاري استخراج كل الإحصائيات بالكامل بدون استثناء...");
    
#     const [resEvent, resLineups] = await Promise.all([
#         fetch(`/api/v1/event/${matchId}`).then(r => r.json()),
#         fetch(`/api/v1/event/${matchId}/lineups`).then(r => r.json())
#     ]);

#     const event = resEvent.event || {};
#     const homeTeam = event.homeTeam?.name || "Home";
#     const awayTeam = event.awayTeam?.name || "Away";
#     const homeScore = event.homeScore?.current || 0;
#     const awayScore = event.awayScore?.current || 0;

#     let rows = [];

#     function processPlayers(players, teamName, isHome) {
#         (players || []).forEach(p => {
#             if (!p.statistics) return;
#             const s = p.statistics;
#             const minutes = s.minutesPlayed || 0;
            
#             // حسابات الالتحامات المجمعة
#             const duelWon = s.duelWon || 0;
#             const duelLost = s.duelLost || 0;
#             const totalDuels = duelWon + duelLost;

#             const aerWon = s.aerialWon || s.aerialDuelWon || 0;
#             const aerLost = s.aerialLost || s.aerialDuelLost || 0;
#             const aerTotal = aerWon + aerLost;

#             const grdWon = Math.max(0, duelWon - aerWon);
#             const grdTotal = Math.max(0, totalDuels - aerTotal);

#             // الكلين شيت
#             const conceded = isHome ? awayScore : homeScore;
#             const cleanSheet = (conceded === 0 && minutes >= 60 && ['G', 'D'].includes(p.position)) ? 1 : 0;

#             // 1. البيانات الأساسية للجدول والعرض
#             const baseData = {
#                 match_id: matchId,
#                 player_id: p.player?.id || null,
#                 player_name: p.player?.name || "",
#                 team: teamName,
#                 position: p.position || "",
#                 minutes_played: minutes,
#                 sofascore_rating: s.rating || 0.0,
#                 clean_sheet: cleanSheet,
#                 // صياغة الأعمدة النصية زي واجهة الموقع
#                 display_passes: `${s.accuratePass || 0}/${s.totalPass || 0}`,
#                 display_total_duels: `${totalDuels} (${duelWon})`,
#                 display_ground_duels: `${grdTotal} (${grdWon})`,
#                 display_aerial_duels: `${aerTotal} (${aerWon})`,
#                 ground_duels_won: grdWon,
#                 ground_duels_total: grdTotal,
#                 aerial_duels_won: aerWon,
#                 aerial_duels_total: aerTotal
#             };

#             // 2. دمج كل إحصائيات سوفاسكور الخام تلقائياً (بما فيها كل الأرقام اللي بعتها)
#             // حذف الكائنات المتداخلة غير الرقمية لتسهيل تحويلها لـ DataFrame
#             const rawStats = { ...s };
#             delete rawStats.ratingVersions;
#             delete rawStats.statisticsType;

#             rows.push({
#                 ...baseData,
#                 ...rawStats
#             });
#         });
#     }

#     processPlayers(resLineups.home?.players, homeTeam, true);
#     processPlayers(resLineups.away?.players, awayTeam, false);

#     // تنزيل الملف
#     const blob = new Blob([JSON.stringify(rows, null, 2)], { type: 'application/json' });
#     const url = URL.createObjectURL(blob);
#     const a = document.createElement('a');
#     a.href = url;
#     a.download = `match_${matchId}_full_stats.json`;
#     document.body.appendChild(a);
#     a.click();
#     document.body.removeChild(a);
#     console.log(`✓ تم بنجاح! تم استخراج ${rows.length} لاعب بجميع المتغيرات الخام.`);
# })();

In [1]:
import glob
import pandas as pd

# قراءة كل ملفات الـ JSON في الفولدر الحالي
json_files = glob.glob("match_*.json")
if not json_files:
    json_files = glob.glob("match_*")

print(f"[+] جاري دمج {len(json_files)} ملفات شاملة كل اللاعبين...")

dataframes = []
for file in json_files:
    try:
        df_match = pd.read_json(file)
        dataframes.append(df_match)
    except Exception as e:
        print(f"[-] خطأ في {file}: {e}")

if dataframes:
    # 1. دمج جميع اللاعبين بلا أي حذف أو استبعاد
    df_all = pd.concat(dataframes, ignore_index=True)

    # 2. ملء كل الخانات المفقودة في الأعمدة الرقمية بـ 0
    numeric_cols = df_all.select_dtypes(include=['float64', 'int64']).columns
    df_all[numeric_cols] = df_all[numeric_cols].fillna(0)

    # 3. ملء الخانات النصية الفارغة بشرطة
    df_all = df_all.fillna("-")

    # 4. حفظ الملف المجمع الكامل
    output_filename = "game_week_1.csv"
    df_all.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!")
    print(f"[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): {len(df_all)}")
    print(f"[✓] اسم الملف: {output_filename}")
    display(df_all.head())

[+] جاري دمج 10 ملفات شاملة كل اللاعبين...
[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!
[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): 400
[✓] اسم الملف: game_week_1.csv


,match_id,player_id,player_name,team,position,minutes_played,sofascore_rating,clean_sheet,display_passes,display_total_duels,...,clearanceOffLine,goals,errorLeadToAGoal,hitWoodwork,penaltyFaced,penaltyConceded,penaltyWon,penaltyMiss,penaltySave,ownGoals
0,16738923,918657,Amr Hossam,Wadi Degla,G,90,8.3,1,17/20,1 (1),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,16738923,2213960,Shady Maher,Wadi Degla,D,90,7.8,1,43/47,10 (6),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,16738923,1085148,Ragab El Safi,Wadi Degla,D,90,7.4,1,67/73,7 (5),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,16738923,919020,Seif Teka,Wadi Degla,D,90,7.3,1,73/81,7 (5),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,16738923,967004,Ahmed Ayman,Wadi Degla,D,73,6.4,1,18/25,8 (2),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


تاكيدات

In [2]:
print("عدد الماتشات:", df_all['match_id'].nunique())
print("عدد الفرق:", df_all['team'].nunique())
print("قائمة الفرق:\n", df_all['team'].unique())

عدد الماتشات: 10
عدد الفرق: 20
قائمة الفرق:
 <StringArray>
[            'Wadi Degla',                 'ZED FC',             'Zamalek SC',
  'Al-Ittihad Alexandria', 'Abu Qir Fertilizers SC', 'National Bank of Egypt',
            'El Gouna FC',        'Modern Sport FC',       'Tala'ea El-Gaish',
    'Arab Contractors FC',    'Ghazl El Mahalla FC',            'Pyramids FC',
           'Cleopatra FC',               'Canal SC',               'Al-Masry',
                 'Smouha',             'Al Ahly FC',    'El-Sharkia Enppi FC',
        'Asyut Petroleum',       'Suez Petrojet FC']
Length: 20, dtype: str


In [3]:
# فحص أقصى وأقل عدد دقائق
print("أقصى دقائق:", df_all['minutes_played'].max())
print("أقل دقائق:", df_all['minutes_played'].min())

# عدد اللاعبين الأساسيين والبدلاء
starters = df_all[df_all['minutes_played'] >= 45]
bench_unplayed = df_all[df_all['minutes_played'] == 0]

print(f"عدد اللي لعبوا 45 دقيقة فأكثر: {len(starters)}")
print(f"عدد اللي قعدوا دكة بدون مشاركة: {len(bench_unplayed)}")

أقصى دقائق: 90
أقل دقائق: 0
عدد اللي لعبوا 45 دقيقة فأكثر: 231
عدد اللي قعدوا دكة بدون مشاركة: 85


In [4]:
# اعرض هدافي الجولة بالكامل للتأكد من الأسماء
top_scorers = df_all[df_all['goals'] > 0][['player_name', 'team', 'goals', 'minutes_played']]
print(top_scorers)

             player_name                    team  goals  minutes_played
93           Ali Hussein  Abu Qir Fertilizers SC    1.0              32
134           Ahmed Zaki             El Gouna FC    1.0               8
150      Mahmoud Mamdouh         Modern Sport FC    1.0              90
161     Ahmed Alaa Eldin        Tala'ea El-Gaish    1.0              90
227        Odeh Fakhoury             Pyramids FC    1.0              66
229       Mahmoud Zalaka             Pyramids FC    1.0              76
231         Nasser Maher             Pyramids FC    1.0              24
248         Mohamed Reda            Cleopatra FC    1.0              90
262         Ibrahim Samy                Canal SC    1.0              90
272          Mawuli Wayo                Canal SC    1.0              29
327         Marwan Attia              Al Ahly FC    1.0              90
330                 Zizo              Al Ahly FC    1.0              86
334  Mohamed Magdy Afsha              Al Ahly FC    1.0         

In [5]:
duplicates = df_all.duplicated(subset=['match_id', 'player_id']).sum()
print("عدد السجلات المكررة:", duplicates) # المفروض يطلع 0

عدد السجلات المكررة: 0


In [6]:
print(df_all['display_passes'].head(30))

0     17/20
1     43/47
2     67/73
3     73/81
4     18/25
5     28/35
6     13/19
7     11/18
8     10/14
9     15/17
10      4/5
11      7/7
12      3/4
13      3/4
14      2/2
15      0/0
16      0/0
17      0/0
18      0/0
19      0/0
20    18/27
21    16/21
22    11/20
23    24/32
24    20/27
25    23/28
26    12/17
27    12/17
28    11/14
29    13/18
Name: display_passes, dtype: str
